# Comparison of our influence metric: linear vs. non-linear

The goal of this notebook is to compare our influence metric from the linear case to the non-linear case.

In [ ]:
import statsmodels.api as sm
from statsmodels.stats import outliers_influence
from statsmodels.formula.api import ols

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

import shap

import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
from main_iom import *

In [ ]:
sns.set_theme(style='white')

## OLS

We read in the data, fit the linear model and calculate SHAP values. 

In [ ]:
data = sm.datasets.get_rdataset("Duncan", "carData").data

In [ ]:
mod = ols("prestige ~ income", data=data).fit()

Caluclate Cook's distance and $p$-value.

In [ ]:
pd.concat([outliers_influence.OLSInfluence(mod).cooks_distance[0], 
           pd.Series(outliers_influence.OLSInfluence(mod).cooks_distance[1], name='pval', index=data.index)], axis=1) \
                .sort_values(['pval'])

Do not reject that there are no influential outliers at the $\alpha=0.05$ confidence level.

In [ ]:
X = data[['income']]
y = data['prestige']

# a simple linear model
model = LinearRegression()
model.fit(X, y)

In [ ]:
sns.regplot(data, x="income", y="prestige", label='Original', fit_reg=True, ci=0)
sns.regplot(data.drop("conductor"), x="income", y="prestige", label='Without Conductor', fit_reg=True, ci=0)
plt.annotate("Conductor", (data.loc["conductor", "income"]-5, data.loc["conductor", "prestige"]), fontsize=12)
plt.annotate("Minister", (data.loc["minister", "income"], data.loc["minister", "prestige"]), fontsize=12)
plt.annotate("RR.engineer", (data.loc["RR.engineer", "income"]-15, data.loc["RR.engineer", "prestige"]), fontsize=12)

plt.legend()

Calculate the residuals.

In [ ]:
resid = np.array(y - model.predict(X))

And the SHAP values.

In [ ]:
explainer = shap.LinearExplainer(model, X)
shap_values = explainer(X).values

## Random Forest

We fit the data with the Random Forest and calculate SHAP values.

In [ ]:
clf = RandomForestRegressor(max_depth=2, random_state=0)
clf.fit(X, y)

In [ ]:
clf_2 = RandomForestRegressor(max_depth=2, random_state=0)
clf_2.fit(X.drop("conductor"), y.drop("conductor"))

In [ ]:
Pred = pd.concat([X.reset_index(drop=True), pd.Series(clf.predict(X))], axis=1, ignore_index=True).sort_values(0)
Pred_2 = pd.concat([X.drop("conductor").reset_index(drop=True), pd.Series(clf_2.predict(X.drop("conductor")))], axis=1, ignore_index=True).sort_values(0)

In [ ]:
sns.scatterplot(data, x="income", y="prestige")
plt.annotate("Conductor", (data.loc["conductor", "income"]-5, data.loc["conductor", "prestige"]), fontsize=12)
plt.annotate("Minister", (data.loc["minister", "income"], data.loc["minister", "prestige"]), fontsize=12)
plt.annotate("RR.engineer", (data.loc["RR.engineer", "income"]-15, data.loc["RR.engineer", "prestige"]), fontsize=12)
sns.lineplot(Pred, x=0, y=1, legend='brief', label='With Conductor')
sns.lineplot(Pred_2, x=0, y=1, legend='brief', label='Without Conductor')
plt.legend()


Calculate residuals.

In [ ]:
resid_rf = np.array(y - clf.predict(X))

And SHAP values.

In [ ]:
explainer_rf = shap.TreeExplainer(clf, X)
shap_values_rf = explainer_rf.shap_values(X)

## Influential outlier metric

We calculate our influence metric for the linear case and the non-linear case.

In [ ]:
print(f"SHAP values: {stats.shapiro(shap_values.reshape(-1)).pvalue:.4f}")
print(f"Residuals: {stats.shapiro(resid.reshape(-1)).pvalue:.4f}")

In [ ]:
iom_lr = InfluentialOutlierMetric(shap_values, resid,
                                  1, 2, 2, 
                                  1, 2, 2, 
                                  lambdas=np.concatenate([[0], np.exp(np.linspace(-1, 10, 50))]),
                                  lambdas_resid=np.concatenate([[0], np.exp(np.linspace(1, 10, 50))]),
                                  epoch=500, epoch_resid=500)

In [ ]:
# Tune for lambda
iom_lr.find_best_lambda(alpha=0.05)
iom_lr.find_best_lambda_resid(alpha=0.05)

# Compute thresholds
thresholds = iom_lr.find_threshold(alpha=[0.05, 0.01])

# Compute IOM
IOM_lr = iom_lr.IOM()

### OLS

In [ ]:
IOM_lr.index = data.index
print(f"Threshold at 0.05: {thresholds[0]:.4f}")
print(f"Threshold at 0.01: {thresholds[1]:.4f}")
IOM_lr.sort_values(ascending=False).round(2)

In [ ]:
iom_lr.summary()

#### Cook's vs. IOM

In [ ]:
cook_iom = pd.concat([pd.Series(outliers_influence.OLSInfluence(mod).cooks_distance[0], name='Cook'), 
                      IOM_lr], axis=1)

In [ ]:
sns.scatterplot(cook_iom, x='Cook', y='IOM')
plt.annotate("Conductor", (cook_iom.loc["conductor", "Cook"]-0.05, cook_iom.loc["conductor", "IOM"]), fontsize=12)
plt.annotate("Minister", (cook_iom.loc["minister", "Cook"]-0.04, cook_iom.loc["minister", "IOM"]), fontsize=12)
plt.annotate("RR.engineer", (cook_iom.loc["RR.engineer", "Cook"], cook_iom.loc["RR.engineer", "IOM"]), fontsize=12)
plt.axhline(thresholds[0], color='green', linestyle='--', label='IOM threshold (α=0.05)')
plt.axhline(thresholds[1], color='red', linestyle='--', label='IOM threshold (α=0.01)')
plt.axvline(4/len(cook_iom), color='blue', linestyle='--', label="Cook's threshold $(4/n)$")
plt.legend()

### Random forest

We assess normality.

In [ ]:
print(f"SHAP values: {stats.shapiro(shap_values_rf.reshape(-1)).pvalue:.4f}")
print(f"Residuals: {stats.shapiro(resid_rf.reshape(-1)).pvalue:.4f}")

We reject the null hypothesis of normality. Now we calculate our IOM.

In [ ]:
iom_rf = InfluentialOutlierMetric(shap_values, resid,
                                  2, 1, 2, 
                                  2, 1, 2, 
                                  lambdas=np.concatenate([[0], np.exp(np.linspace(-5, 5, 50))]),
                                  lambdas_resid=np.concatenate([[0], np.exp(np.linspace(1, 10, 50))]),
                                  epoch=500, epoch_resid=500)

In [ ]:
# Tune for lambda
iom_rf.find_best_lambda(alpha=0.05)
iom_rf.find_best_lambda_resid(alpha=0.05)

# Compute thresholds
thresholds = iom_rf.find_threshold(alpha=[0.05, 0.01])

# Compute IOM
IOM_rf = iom_rf.IOM()

In [ ]:
IOM_rf.index = data.index
print(f"Threshold at 0.05: {thresholds[0]:.4f}")
print(f"Threshold at 0.01: {thresholds[1]:.4f}")
IOM_rf.sort_values(ascending=False).round(2)

In [ ]:
iom_rf.summary()